# 04 — Generation, GPT-5 Judge, Abstention, and Validation

Generator backbone selection (Table 23), end-to-end RAG (Tables 24–25), abstention
(Table 26), and judge validation against GPT-4o and three human experts (Tables 27–28).

**Models:** generator = GPT-4.1; primary judge = GPT-5; secondary judge = GPT-4o.

> NOTE: prompts here are the SHORT prompts actually used in the run (see cell below).
> Keep paper Appendix A in sync with these.


In [ ]:
# Configuration is centralised in src/config.py (single source of truth).
# If running outside the package, the constants below mirror that file.
import sys, os
sys.path.append(os.path.abspath(".."))
try:
    from src.config import *
    print("Loaded config from src.config")
except Exception as e:
    print("src.config not importable here; using inline constants.", e)
    RANDOM_SEED = 42
    RRF_KAPPA = 60
    DENSE_BATCH_SIZE = 128
    MAX_SEQ_LENGTH = 512
    NAIVE_K, STATIC_K = 5, 10
    ADAPTIVE_K = {"simple":5,"moderate":10,"complex":15}


## Generation configuration, prompts, backends

In [ ]:
#@title 2.1 Generator comparison — configuration (all 242: answerable + unanswerable)
import gc
RUN_GENERATION = True      # set True to generate (GPU + API budget)
USE_ALL = True                # True = full benchmark (242). False = stratified subset.
GEN_EVAL_N = 90               # used only when USE_ALL=False
GEN_MAX_NEW_TOKENS = 512

GEN_SYSTEMS = [
    ('KazLLM-1.0-8B',         'issai/LLama-3.1-KazLLM-1.0-8B',       'hf',     'LLM-only', 'kz'),
    ('Hermes-3-Llama-3.1-8B', 'NousResearch/Hermes-3-Llama-3.1-8B',  'hf',     'LLM-only', 'kz'),
    ('Qwen2.5-7B',            'Qwen/Qwen2.5-7B-Instruct',            'hf',     'LLM-only', 'kz'),
    ('GPT-4.1 (LLM-only)',    'gpt-4.1',                             'openai', 'LLM-only', 'kz'),
    ('GPT-4.1 (RAG KZ)',      'gpt-4.1',                             'openai', 'RAG',      'kz'),
    ('GPT-4.1 (RAG RUS)',     'gpt-4.1',                             'openai', 'RAG',      'rus'),
]

# ---- Build the evaluation set ----
if USE_ALL:
    eval_subset = bench_all.copy().reset_index(drop=True)
else:
    _pool = bench_all.copy()
    per = max(1, GEN_EVAL_N // _pool['complexity'].nunique())
    eval_subset = (_pool.groupby('complexity', group_keys=False)
                   .apply(lambda s: s.sample(min(len(s), per), random_state=RANDOM_SEED))
                   .reset_index(drop=True))

n_ans = (eval_subset['answerability'] == 'answerable').sum()
n_un  = (eval_subset['answerability'] == 'unanswerable').sum()
print(f'Eval subset: {len(eval_subset)} questions ({n_ans} answerable + {n_un} unanswerable)')

# ---- Evidence lookup: Static retrieval for EVERY row (not just 182) ----
if 'static_all_res' in dir() and len(static_all_res) == len(bench_all):
    _retr = {rid: static_all_res[i] for i, rid in enumerate(bench_all['row_id'].tolist())}
else:
    print('Building Static Hybrid retrieval for all 242 (needed for unanswerable evidence)...')
    if 'query_embs_all' not in dir() or len(query_embs_all) != len(bench_all):
        query_embs_all = dense_model.encode(bench_all['question'].tolist(),
                          batch_size=DENSE_BATCH_SIZE, show_progress_bar=True,
                          convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    _retr = {}
    for i, row in tqdm(list(bench_all.iterrows()), desc='Static retrieval (242)'):
        _retr[row['row_id']] = static_hybrid_retrieve(row['question'], query_embs_all[i], k=STATIC_K)

def evidence_for_row(row):
    res = _retr.get(row['row_id'])
    return evidence_context_from_result(res, max_chunks=10) if res is not None else ''

# ---- Prompts ----
PROMPTS = {
 'kz': {'LLM-only':"Қазақ тіліндегі заң сұрағына нақты жауап беріңіз. Сенімді болмасаңыз, анықтау мүмкін емес деп жазыңыз.\n\nСұрақ:\n{q}",
        'RAG':"Берілген дереккөздерді пайдаланып қазақ тіліндегі заң сұрағына жауап беріңіз. Дереккөз жеткіліксіз болса 'Берілген дереккөздерде бұл сұраққа жеткілікті негіз жоқ.' деп бас тартыңыз. [E1],[E2] сілтеме жасаңыз.\n\nСұрақ:\n{q}\n\nДереккөздер:\n{e}"},
 'rus':{'LLM-only':"Дайте точный ответ на юридический вопрос. Если не уверены, укажите что ответ не может быть определён.\n\nВопрос:\n{q}",
        'RAG':"Используя источники, ответьте на юридический вопрос. Если их недостаточно — откажитесь. Цитируйте [E1],[E2].\n\nВопрос:\n{q}\n\nИсточники:\n{e}"},
}
def build_prompt(setting, lang, q, e=''):
    return PROMPTS[lang][setting].format(q=q, e=e)

In [ ]:
#@title 2.2 Backends — local HF (fp16, no bitsandbytes) + OpenAI generator
import gc, torch
_HF = {}
def load_hf_fp16(mid):
    """Load an HF causal LM in fp16. A100/RTX PRO 6000 has enough VRAM; no 4-bit needed."""
    if mid in _HF: return _HF[mid]
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        mid, device_map='auto', trust_remote_code=True, dtype=torch.float16).eval()
    _HF[mid] = (tok, model); return tok, model

def free_hf(mid):
    if mid in _HF: del _HF[mid]
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def gen_hf(mid, prompt, max_new=GEN_MAX_NEW_TOKENS):
    tok, model = load_hf_fp16(mid)
    try:
        text = tok.apply_chat_template([{'role':'user','content':prompt}],
                                       add_generation_prompt=True, tokenize=False)
    except Exception:
        text = prompt
    enc = tok(text, return_tensors='pt', truncation=True, max_length=4096)
    enc = {k: v.to(model.device) for k,v in enc.items()}
    ilen = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ilen:], skip_special_tokens=True).strip()

# OpenAI generator (GPT-4.1). Requires OPENAI_API_KEY in env.
def gen_openai(mid, prompt, max_new=GEN_MAX_NEW_TOKENS):
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(model=mid, temperature=0, max_tokens=max_new,
            messages=[{'role':'user','content':prompt}])
    return r.choices[0].message.content.strip()

print('Backends ready: gen_hf (fp16), gen_openai (GPT-4.1).')


In [ ]:
#@title 2.3 Run generation across all systems (fp16 locals + GPT-4.1)
if RUN_GENERATION:
    # sanity: one fp16 generation must work before the long loop
    _t = gen_hf('Qwen/Qwen2.5-7B-Instruct', 'Сәлем')
    assert not str(_t).startswith('GEN_FAILED'), f'fp16 broken: {_t[:200]}'
    free_hf('Qwen/Qwen2.5-7B-Instruct')
    print('fp16 confirmed:', repr(_t[:50]), '\n')

    recs = []
    for name, mid, backend, setting, lang in GEN_SYSTEMS:
        print(f'=== {name} | {backend} | {setting} | {lang} ===')
        for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc=name):
            ev = evidence_for_row(row) if setting=='RAG' else ''
            try:
                prompt = build_prompt(setting, lang, row['question'], ev)
                ans = gen_hf(mid, prompt) if backend=='hf' else gen_openai(mid, prompt)
            except Exception as e:
                ans = f'GEN_FAILED: {type(e).__name__}: {e}'
            recs.append({'system':name,'model_id':mid,'backend':backend,'setting':setting,'language':lang,
                         'row_id':row['row_id'],'question':row['question'],
                         'gold_answer':row.get('gold_answer',''),'gold_chunk_id':row.get('gold_chunk_id_str',''),
                         'generated_answer':ans})
        if backend=='hf': free_hf(mid)
    gen_compare_df = pd.DataFrame(recs)
    save_table(gen_compare_df, 'gen_compare_raw')
    print()
    for s,sub in gen_compare_df.groupby('system'):
        f=sub['generated_answer'].astype(str).str.startswith('GEN_FAILED').sum()
        print(f'  {s:25} OK={len(sub)-f} FAILED={f}')
else:
    print('RUN_GENERATION is False — set it True to generate.')
    gen_compare_df = pd.DataFrame()


## Metrics (surface + semantic) and GPT-5 judge (primary)

In [ ]:
#@title 3.0 Full metrics — surface (EM/F1/chrF++/ROUGE-L) + semantic (BGE-M3 cosine)
import re, string
from collections import Counter
try:
    from sacrebleu.metrics import CHRF
except Exception:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','sacrebleu'])
    from sacrebleu.metrics import CHRF
_chrf = CHRF(word_order=2)   # chrF++

amap = dict(zip(bench_all['row_id'], bench_all['answerability']))
_PUNCT=set(string.punctuation)
def _norm(s):
    s=str(s).lower(); s=''.join(c for c in s if c not in _PUNCT)
    return re.sub(r'\s+',' ',s).strip()
def _toks(s): return _norm(s).split()
def exact_match(p,g): return int(_norm(p)==_norm(g))
def squad_f1(p,g):
    pt,gt=_toks(p),_toks(g)
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    ov=sum((Counter(pt)&Counter(gt)).values())
    if ov==0: return 0.0
    pr=ov/len(pt); rc=ov/len(gt); return 2*pr*rc/(pr+rc)
def rouge_l(p,g):
    a,b=_toks(p),_toks(g)
    if not a or not b: return 0.0
    m,n=len(a),len(b); dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j]=dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j],dp[i][j-1])
    l=dp[m][n]
    if l==0: return 0.0
    pr=l/len(a); rc=l/len(b); return 2*pr*rc/(pr+rc)
ABSTAIN=['жеткілікті негіз жоқ','жауап табылған жоқ','жауап жоқ','мүмкін емес','недостаточно',
         'не найден','не могу','нет информации','cannot','insufficient','no answer','нет ответа']
def is_abstain(t): t=str(t).lower(); return int(any(m in t for m in ABSTAIN))
def is_failed(t): return str(t).startswith('GEN_FAILED')

assert 'gen_compare_df' in dir() and len(gen_compare_df), 'Run generation first.'
df = gen_compare_df.copy()
df['answerability'] = df['row_id'].map(amap)

em=[];f1=[];chrf=[];rl=[];abst=[]
for _,r in df.iterrows():
    a=str(r['generated_answer']); g=str(r.get('gold_answer','')); av=r['answerability']; failed=is_failed(a)
    abst.append(is_abstain(a))
    if av=='answerable' and not failed:
        em.append(exact_match(a,g)); f1.append(squad_f1(a,g))
        chrf.append(float(_chrf.sentence_score(a,[g]).score)); rl.append(rouge_l(a,g))
    else:
        em.append(np.nan); f1.append(np.nan); chrf.append(np.nan); rl.append(np.nan)
df['EM']=em; df['F1']=f1; df['chrf']=chrf; df['rouge_l']=rl; df['abstained']=abst

# semantic similarity (answerable, non-failed)
df['semantic_sim']=np.nan
mask = df['answerability'].eq('answerable') & ~df['generated_answer'].apply(is_failed)
sub = df[mask]
if len(sub):
    ep = dense_model.encode(sub['generated_answer'].astype(str).tolist(), batch_size=DENSE_BATCH_SIZE,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    eg = dense_model.encode(sub['gold_answer'].astype(str).tolist(), batch_size=DENSE_BATCH_SIZE,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    df.loc[sub.index,'semantic_sim']=(ep*eg).sum(axis=1)
gen_compare_df = df
save_table(gen_compare_df, 'gen_compare_with_metrics')
print('Surface + semantic metrics written into gen_compare_df.')


In [ ]:
#@title 3.1 GPT-5 LLM-judge (Section 3.7.5) — legal-grounding scores
# NOTE: generator GPT-4.1 and judge GPT-5 share a provider family -> possible
# self-preference bias. Report this in Limitations. chrF++/semantic stay objective.
RUN_JUDGE = True
JUDGE_MODEL = 'gpt-5'   # pin exact snapshot + access date in the paper
import json as _json, re as _re
JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER to the GOLD ANSWER for the QUESTION.
Score each dimension as an integer 0-2 (hallucination: 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}"""
def judge_one(q,g,a):
    try:
        from openai import OpenAI
        client=OpenAI()
        c=JUDGE_PROMPT.format(question=q,gold=g,answer=a)
        try:
            r=client.chat.completions.create(model=JUDGE_MODEL,temperature=0,
                messages=[{'role':'user','content':c}])
        except Exception:
            r=client.chat.completions.create(model=JUDGE_MODEL,
                messages=[{'role':'user','content':c}])
        t=_re.sub(r'```json|```','',r.choices[0].message.content.strip()).strip()
        return _json.loads(t)
    except Exception as e:
        return {'_error':f'{type(e).__name__}: {e}'}

DIMS=['legal_correctness','evidence_support','completeness','clarity','hallucination']
for d in DIMS: gen_compare_df[f'judge_{d}']=np.nan
if RUN_JUDGE:
    jmask = gen_compare_df['answerability'].eq('answerable') & ~gen_compare_df['generated_answer'].apply(lambda t: str(t).startswith('GEN_FAILED'))
    for i in tqdm(gen_compare_df[jmask].index, desc='GPT-5 judge'):
        r=gen_compare_df.loc[i]
        s=judge_one(r['question'], r['gold_answer'], r['generated_answer'])
        if '_error' not in s:
            for d in DIMS: gen_compare_df.loc[i,f'judge_{d}']=s.get(d,np.nan)
    save_table(gen_compare_df, 'gen_compare_judged_gpt5')
    print('GPT-5 judge scores written.')
else:
    print('RUN_JUDGE is False.')


## Stage 1 — Generator backbone selection (Table 23)

In [ ]:
#@title 4.1 STAGE 1 — Generator backbone selection (LLM-only, answerable)
# Pick the strongest base LLM in the SAME condition (LLM-only, no RAG).
# Selection priority: Legal correctness (judge) > SemanticSim > chrF++.
_fail = lambda t: str(t).startswith('GEN_FAILED')
s1 = gen_compare_df.copy()
s1 = s1[(s1['setting']=='LLM-only') & (s1['answerability']=='answerable') & ~s1['generated_answer'].apply(_fail)]
rows=[]
for sysn, g in s1.groupby('system'):
    rows.append({'Model':sysn,'N':len(g),
        'chrF++':round(float(g['chrf'].mean()),2),
        'F1':round(float(g['F1'].mean()),3),
        'ROUGE-L':round(float(g['rouge_l'].mean()),3),
        'SemanticSim':round(float(g['semantic_sim'].mean()),3),
        'Legal':round(float(g['judge_legal_correctness'].mean()),3) if g['judge_legal_correctness'].notna().any() else None,
        'Evidence':round(float(g['judge_evidence_support'].mean()),3) if g['judge_evidence_support'].notna().any() else None,
        'Halluc':round(float(g['judge_hallucination'].mean()),3) if g['judge_hallucination'].notna().any() else None})
backbone_table=pd.DataFrame(rows)
sort_cols=[c for c in ['Legal','SemanticSim','chrF++'] if c in backbone_table and backbone_table[c].notna().any()]
backbone_table=backbone_table.sort_values(sort_cols,ascending=False).reset_index(drop=True)
best_model=backbone_table.iloc[0]['Model']
display(backbone_table)
save_table(backbone_table,'tableA_generator_backbone_selection')
print(f'\n>>> BEST GENERATOR (Stage 1): {best_model}')
print('Priority: Legal > SemanticSim > chrF++. If best != GPT-4.1, generate its RAG variants for Stage 2.')


## Stage 2 — End-to-end RAG + abstention (Tables 24–26)

> CHECK: in cell 25 the 'Citation' column reads `judge_completeness`; verify it should be
> `judge_citation_accuracy`. Fix before publishing if the released numbers were affected.

In [ ]:
#@title 4.2 STAGE 2 — End-to-end RAG (best generator) + Table 22 + abstention
# Compares the chosen generator across settings: LLM-only vs Static RAG KZ vs RAG RUS.
# Quality (legal/evidence/semantic) on answerable; abstention on the 60 unanswerable.
_fail = lambda t: str(t).startswith('GEN_FAILED')
gdf = gen_compare_df.copy()

# Table 22 — per (system,setting) quality on answerable
rows=[]
for (sysn,setting), sub in gdf.groupby(['system','setting']):
    a = sub[(sub['answerability']=='answerable') & ~sub['generated_answer'].apply(_fail)]
    u = sub[sub['answerability']=='unanswerable']
    rows.append({'System':sysn,'Setting':setting,'N_ans':len(a),'N_unans':len(u),
        'chrF++':round(float(a['chrf'].mean()),2) if len(a) else None,
        'F1':round(float(a['F1'].mean()),3) if len(a) else None,
        'SemanticSim':round(float(a['semantic_sim'].mean()),3) if len(a) else None,
        'Legal':round(float(a['judge_legal_correctness'].mean()),3) if 'judge_legal_correctness' in a and a['judge_legal_correctness'].notna().any() else None,
        'Evidence':round(float(a['judge_evidence_support'].mean()),3) if 'judge_evidence_support' in a and a['judge_evidence_support'].notna().any() else None,
        'Citation':round(float(a['judge_completeness'].mean()),3) if 'judge_completeness' in a and a['judge_completeness'].notna().any() else None,
        'Halluc':round(float(a['judge_hallucination'].mean()),3) if 'judge_hallucination' in a and a['judge_hallucination'].notna().any() else None,
        'CorrectAbstain':round(float(u['abstained'].mean()),3) if len(u) else None,   # higher better
        'FalseAbstain':round(float(a['abstained'].mean()),3) if len(a) else None})    # lower better
table22 = pd.DataFrame(rows)
display(table22); save_table(table22,'table22_end_to_end_generation')

# Abstention precision/recall/F1 (refusal detection on unanswerable)
arows=[]
for (sysn,setting), sub in gdf.groupby(['system','setting']):
    u=sub[sub['answerability']=='unanswerable']; an=sub[sub['answerability']=='answerable']
    tp=int(u['abstained'].sum()); fn=len(u)-tp; fp=int(an['abstained'].sum())
    prec=tp/(tp+fp) if (tp+fp) else None
    rec =tp/(tp+fn) if (tp+fn) else None
    f1=2*prec*rec/(prec+rec) if (prec and rec) else None
    arows.append({'System':sysn,'Setting':setting,
        'Abstain_P':round(prec,3) if prec is not None else None,
        'Abstain_R':round(rec,3) if rec is not None else None,
        'Abstain_F1':round(f1,3) if f1 is not None else None})
abst_table=pd.DataFrame(arows)
display(abst_table); save_table(abst_table,'table_abstention_metrics')
print('Table 22 = end-to-end generation quality; abstention = refusal detection on the 60 unanswerable.')
print('Judge = GPT-5. Note self-preference caveat (GPT-4.1 generator) in Limitations.')


## Secondary judge (GPT-4o) cross-check + judge validation (Section 4.10.4)

In [ ]:
#@title 10.3f LLM-judge scoring (GPT-4o) + automatic metrics
JUDGE_MODEL = 'gpt-4o'   # pin exact snapshot in the paper (model + access date)
RUN_JUDGE = False        # set True after generation; uses API budget

import re as _re
try:
    import sacrebleu
    def chrf_pp(hyp, ref):
        if not str(ref).strip():
            return np.nan
        return sacrebleu.sentence_chrf(str(hyp), [str(ref)],
                                       word_order=2).score   # chrF++
except Exception:
    def chrf_pp(hyp, ref):
        return np.nan

def token_f1(hyp, ref):
    h = set(str(hyp).lower().split()); r = set(str(ref).lower().split())
    if not h or not r:
        return 0.0
    tp = len(h & r)
    if tp == 0:
        return 0.0
    p = tp / len(h); rc = tp / len(r)
    return 2 * p * rc / (p + rc)

JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER against the GOLD ANSWER for the QUESTION.
Score each dimension on an integer 0-2 scale (hallucination is 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}
"""

def judge_one(question, gold, answer):
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=JUDGE_MODEL, temperature=0,
            messages=[{'role': 'user',
                       'content': JUDGE_PROMPT.format(question=question, gold=gold, answer=answer)}])
        txt = resp.choices[0].message.content.strip()
        txt = _re.sub(r'```json|```', '', txt).strip()
        return json.loads(txt)
    except Exception as e:
        return {'error': f'{type(e).__name__}: {e}'}

if RUN_JUDGE and len(gen_compare_df):
    judged = []
    for _, r in tqdm(gen_compare_df.iterrows(), total=len(gen_compare_df), desc='LLM-judge'):
        scores = judge_one(r['question'], r['gold_answer'], r['generated_answer'])
        rec = {**r.to_dict()}
        for dim in ['legal_correctness', 'evidence_support', 'completeness', 'clarity', 'hallucination']:
            rec[f'judge_{dim}'] = scores.get(dim, np.nan) if 'error' not in scores else np.nan
        rec['chrF++'] = chrf_pp(r['generated_answer'], r['gold_answer'])
        rec['token_f1'] = token_f1(r['generated_answer'], r['gold_answer'])
        judged.append(rec)
    judged_df = pd.DataFrame(judged)
    save_table(judged_df, 'table14_generator_comparison_judged')

    summary = (judged_df.groupby(['system', 'setting', 'language'])
               .agg(N=('row_id', 'size'),
                    legal_correctness=('judge_legal_correctness', 'mean'),
                    evidence_support=('judge_evidence_support', 'mean'),
                    completeness=('judge_completeness', 'mean'),
                    clarity=('judge_clarity', 'mean'),
                    hallucination=('judge_hallucination', 'mean'),
                    chrFpp=('chrF++', 'mean'),
                    token_f1=('token_f1', 'mean'),
                    latency_s=('latency_s', 'mean'))
               .round(3).reset_index())
    display(summary)
    save_table(summary, 'table14_generator_comparison_summary')
    print('Note for paper: GPT-4.1 LLM-only is the shared anchor — compare it against '
          'GPT-4.1 RAG to isolate the retrieval contribution, and against the local '
          'LLM-only models to isolate the backbone contribution. '
          'Judge model and generator share the GPT family: report self-preference bias as a limitation, '
          'ideally cross-check a subset with a non-GPT judge.')
else:
    print('RUN_JUDGE is False or no generations available.')


In [ ]:
#@title 10.3h Judge validation — GPT-4o vs GPT-5 agreement (kappa + system-ranking Spearman)
# Validates the LLM judge: do two different judges agree per-item (weighted kappa /
# Krippendorff alpha) and do they induce the SAME system ranking (Spearman)?
# Requires gen_compare_df (from 10.3e) with real generated answers.
import re as _rej, json as _json, time as _t
import numpy as np, pandas as pd
from tqdm.auto import tqdm

RUN_JUDGE_VALIDATION = False           # set True; calls BOTH judges on every answer (2x API cost)
JUDGE_MODELS = {'judgeA': 'gpt-4o', 'judgeB': 'gpt-5'}   # pin exact snapshots + access date in paper
JUDGE_DIMS = ['legal_correctness', 'evidence_support', 'completeness', 'clarity', 'hallucination']

_JUDGE_PROMPT = """You are a strict bilingual (Kazakh/Russian) legal QA evaluator.
Compare the SYSTEM ANSWER against the GOLD ANSWER for the QUESTION.
Score each dimension on an integer 0-2 scale (hallucination: 0=none,1=minor,2=major).
Treat Kazakh morphological variants as equivalent. Return ONLY JSON:
{{"legal_correctness":0-2,"evidence_support":0-2,"completeness":0-2,"clarity":0-2,"hallucination":0-2}}

QUESTION:
{question}

GOLD ANSWER:
{gold}

SYSTEM ANSWER:
{answer}
"""

def _judge_call(model_id, question, gold, answer):
    try:
        from openai import OpenAI
        client = OpenAI()
        kwargs = dict(model=model_id,
                      messages=[{'role': 'user',
                                 'content': _JUDGE_PROMPT.format(question=question, gold=gold, answer=answer)}])
        # GPT-5 family may not accept temperature; try with, fall back without.
        try:
            resp = client.chat.completions.create(temperature=0, **kwargs)
        except Exception:
            resp = client.chat.completions.create(**kwargs)
        txt = _rej.sub(r'```json|```', '', resp.choices[0].message.content.strip()).strip()
        return _json.loads(txt)
    except Exception as e:
        return {'_error': f'{type(e).__name__}: {e}'}

if RUN_JUDGE_VALIDATION and 'gen_compare_df' in dir() and len(gen_compare_df):
    recs = []
    for _, r in tqdm(gen_compare_df.iterrows(), total=len(gen_compare_df), desc='Dual-judge'):
        rec = {'system': r['system'], 'setting': r['setting'], 'row_id': r['row_id']}
        for tag, mid in JUDGE_MODELS.items():
            s = _judge_call(mid, r['question'], r['gold_answer'], r['generated_answer'])
            for d in JUDGE_DIMS:
                rec[f'{tag}_{d}'] = s.get(d, np.nan) if '_error' not in s else np.nan
        recs.append(rec)
    dual = pd.DataFrame(recs)
    save_table(dual, 'table_judge_validation_raw')

    # ---- 1) Per-dimension agreement: weighted Cohen kappa + Krippendorff alpha ----
    try:
        from sklearn.metrics import cohen_kappa_score
        have_sk = True
    except Exception:
        have_sk = False

    def krippendorff_alpha_ordinal(a, b):
        # simple 2-rater ordinal alpha
        pairs = [(x, y) for x, y in zip(a, b) if pd.notna(x) and pd.notna(y)]
        if not pairs:
            return np.nan
        vals = [v for p in pairs for v in p]
        n = len(pairs)
        Do = np.mean([(x - y) ** 2 for x, y in pairs])
        gm = np.mean(vals)
        De = np.mean([(v - gm) ** 2 for v in vals]) * 2
        return 1 - Do / De if De else np.nan

    agree_rows = []
    for d in JUDGE_DIMS:
        a = pd.to_numeric(dual[f'judgeA_{d}'], errors='coerce')
        b = pd.to_numeric(dual[f'judgeB_{d}'], errors='coerce')
        mask = a.notna() & b.notna()
        n = int(mask.sum())
        pa = float((a[mask] == b[mask]).mean()) if n else np.nan
        wk = (cohen_kappa_score(a[mask].astype(int), b[mask].astype(int), weights='quadratic')
              if have_sk and n and a[mask].nunique() > 1 and b[mask].nunique() > 1 else np.nan)
        alpha = krippendorff_alpha_ordinal(a[mask].tolist(), b[mask].tolist())
        agree_rows.append({'Dimension': d, 'N': n,
                           'Percent_agreement': round(pa, 3) if pd.notna(pa) else None,
                           'Weighted_kappa': round(wk, 3) if pd.notna(wk) else None,
                           'Krippendorff_alpha': round(alpha, 3) if pd.notna(alpha) else None})
    judge_agreement = pd.DataFrame(agree_rows)
    display(judge_agreement)
    save_table(judge_agreement, 'table_judge_agreement')

    # ---- 2) System-ranking agreement: do both judges rank systems the same? ----
    from scipy.stats import spearmanr
    rankA = dual.groupby('system')[[f'judgeA_{d}' for d in JUDGE_DIMS]].mean().mean(axis=1)
    rankB = dual.groupby('system')[[f'judgeB_{d}' for d in JUDGE_DIMS]].mean().mean(axis=1)
    rank_tbl = pd.DataFrame({'judgeA_mean': rankA, 'judgeB_mean': rankB})
    rank_tbl['rankA'] = rank_tbl['judgeA_mean'].rank(ascending=False)
    rank_tbl['rankB'] = rank_tbl['judgeB_mean'].rank(ascending=False)
    rho, p = spearmanr(rank_tbl['judgeA_mean'], rank_tbl['judgeB_mean'])
    display(rank_tbl.round(3))
    save_table(rank_tbl.reset_index(), 'table_judge_system_ranking')
    print(f'System-ranking Spearman rho = {rho:.3f} (p = {p:.4f}). '
          'High rho => the "which system is best" conclusion is robust to judge choice.')
    print('Report in paper: GPT-5 is the primary judge (Section 3.7.5); GPT-4o cross-check. '
          'State self-preference caveat: GPT-4.1 generations judged by GPT-family judges.')
else:
    print('Set RUN_JUDGE_VALIDATION=True and ensure gen_compare_df has real generated answers.')


## Expert benchmark validation: adjudicated results + IAA (Tables 13–15, 27)

In [ ]:
#@title 11.2 Aggregation from adjudicated (final) labels — adjudicated version
COMPLETED_JUDGE_FILE = '/content/adjudicated_evaluation_100.xlsx'

if COMPLETED_JUDGE_FILE and Path(COMPLETED_JUDGE_FILE).exists():
    # adjudicated file: final scores live on the 'Adjudicated' sheet
    try:
        labeled = pd.read_excel(COMPLETED_JUDGE_FILE, sheet_name='Adjudicated')
    except (ValueError, KeyError):
        labeled = pd.read_excel(COMPLETED_JUDGE_FILE)  # fallback: first sheet

    print('Loaded file:', COMPLETED_JUDGE_FILE)
    print('Columns:', list(labeled.columns))

    # Detect grouping column
    if 'variant' in labeled.columns:
        group_col = 'variant'
    elif 'system' in labeled.columns:
        group_col = 'system'
    elif 'System' in labeled.columns:
        group_col = 'System'
    elif 'Variant' in labeled.columns:
        group_col = 'Variant'
    else:
        labeled['_all_rows'] = 'Adjudicated Human Evaluation'
        group_col = '_all_rows'
        print('No variant/system column found. Aggregating all rows together.')

    # Candidate metric columns — adjudicated file uses the _FINAL suffix.
    # Both suffixed and plain names are listed so the cell also works on raw annotator files.
    metric_candidates = [
        'legal_correctness_0_1_2_FINAL',
        'completeness_0_1_2_FINAL',
        'answer_completeness_0_1_2_FINAL',
        'evidence_support_0_1_2_FINAL',
        'citation_accuracy_0_1_FINAL',
        'clarity_0_1_2_FINAL',
        'hallucination_yes_no_FINAL',
        # plain (raw annotator) fallbacks
        'legal_correctness_0_1_2',
        'answer_completeness_0_1_2',
        'completeness_0_1_2',
        'evidence_support_0_1_2',
        'citation_accuracy_0_1',
        'clarity_0_1_2',
        'hallucination_yes_no',
        'notes',
    ]

    existing_metrics = [c for c in metric_candidates if c in labeled.columns]
    print('Detected metric columns:', existing_metrics)

    if not existing_metrics:
        print(
            'No filled judge/human metric columns detected. '
            'This file is probably still an empty annotation template.'
        )
    else:
        agg_rows = []
        for variant, sub in labeled.groupby(group_col, dropna=False):
            row = {'Variant': variant, 'N': len(sub)}
            for col in existing_metrics:
                if col == 'notes':
                    continue
                if col in ['hallucination_yes_no', 'hallucination',
                           'hallucination_yes_no_FINAL']:
                    row['HallucinationRate'] = round(
                        sub[col].astype(str).str.strip().str.lower().isin(
                            ['yes', 'true', '1', 'иә', 'да']
                        ).mean(),
                        4
                    )
                else:
                    row[col] = round(
                        pd.to_numeric(sub[col], errors='coerce').mean(),
                        4
                    )
            agg_rows.append(row)

        generation_quality_df = pd.DataFrame(agg_rows)
        display(generation_quality_df)
        save_table(
            generation_quality_df,
            'table09_adjudicated_human_quality_from_labels'
        )
else:
    print(
        'No completed judge file configured or file does not exist. '
        'Fill / adjudicate the annotations, then rerun this cell.'
    )

In [ ]:
#@title 11.3 Inter-annotator agreement (pre-adjudication) for the paper
# Reads the Agreement_pre_adjud sheet from the adjudicated workbook.
AGREEMENT_SHEET = 'Agreement_pre_adjud'
try:
    agree = pd.read_excel(COMPLETED_JUDGE_FILE, sheet_name=AGREEMENT_SHEET, header=2)
    agree = agree.dropna(how='all')
    display(agree)
    save_table(agree, 'table10_inter_annotator_agreement')
    print('Reminder: agreement is computed BEFORE adjudication; '
          'final quality scores in 11.2 are computed AFTER adjudication.')
    print('Note for the paper: low/negative kappa with high % agreement reflects the '
          'kappa paradox (one annotator used near-constant max ratings -> low label variance), '
          'not unreliable annotation. Report both % agreement and kappa, and state this explicitly.')
except Exception as e:
    print('Could not load agreement sheet (is adjudicated_evaluation_100.xlsx present?):', e)
